# Pilot loss comparison — Arm A (elicit) vs Arm B (teach)

Compares the OPEN(2) pilot pair on the loss curves the runs actually log
(spec 00 / spec 02 §11):

- **per-step train loss** (`train_log.jsonl` — pre-update masked loss on each
  incoming batch, all epochs),
- **held-out val loss** (`eval_log.jsonl` — the ε/k stopping evals, every 500 steps),
- **the paper-shaped prequential learning curve** (`logs/prequential.jsonl`,
  epoch-1 only, via `geode.edl.metrics.training_curve`) plus MDL/EDL totals.

The arms are identical recipes on the identical 50K D_target prefix; they differ
only in the parent: A (run-3 lineage) had the add/sub **algorithm pre-taught** in
natural-language notation, B (run-4 lineage) got **format only**. D_target must
therefore only *elicit* (remap notation) for A but *teach* the algorithm to B.

**Paper-consistent (Donoway et al. EDL):** A starts lower and converges in far
fewer examples; EDL_A ≪ EDL_B; both end near-zero. The paper's absolute numbers
(1B models, ~300K teaching scale) don't transfer to 38.7M — the *qualitative
separation* is what we check. **If the curves overlap, we are in trouble** — see
the reading guide at the bottom.


In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# repo root: walk up until we see the geode/ package (same pattern as view_dataset)
here = Path.cwd()
REPO_ROOT = next(
    (p for p in [here, *here.parents] if (p / "geode" / "__init__.py").exists()), None
)
assert REPO_ROOT is not None, "run this notebook from inside the elicit-vs-teach checkout"
sys.path.insert(0, str(REPO_ROOT))

from geode.edl.metrics import edl_nats, mdl_nats, nats_to_bits, training_curve  # noqa: E402

# ---- config: edit these ----
STORE = Path(os.environ.get("GEODE_STORE") or REPO_ROOT / "geode-store")
RUNS = {  # label -> run_id; add the other grid points here to overlay them
    "Arm A (elicit) @50K": "evt-run5-pilot-n50k", 
    "Arm B (teach) @50K": "evt-run6-pilot-n50k",
    "Arm B @10K": "evt-run6-pilot-n10k",
    "Arm B @200K": "evt-run6-pilot-n200k",
    "Arm B @500K": "evt-run6-pilot-n500k",
}


In [15]:
def read_jsonl(path: Path) -> pd.DataFrame:
    return pd.DataFrame(json.loads(ln) for ln in path.read_text().splitlines() if ln.strip())


runs, rows = {}, []
for label, rid in RUNS.items():
    d = STORE / "runs" / rid
    try:  # logs/prequential.jsonl lands only when the run finishes
        curve = training_curve(rid, store=STORE)
    except FileNotFoundError:
        curve = None
    m = json.loads((d / "manifest.json").read_text())
    res = m["experiment"].get("target_result") or {}
    runs[label] = {
        "id": rid,
        "train": read_jsonl(d / "train_log.jsonl"),
        "val": read_jsonl(d / "eval_log.jsonl"),
        "curve": curve,
        "result": res,
    }
    rows.append(
        {
            "run": label,
            "run_id": rid,
            "status": m.get("status"),
            "stop_reason": res.get("stop_reason"),
            "final_step": res.get("final_step"),
            "min_val_bits": None
            if res.get("min_val_nats") is None
            else nats_to_bits(res["min_val_nats"]),
        }
    )
pd.DataFrame(rows)


FileNotFoundError: [Errno 2] No such file or directory: '~/Github/geode/geode-store/runs/evt-run5-pilot-n50k/manifest.json'

In [ ]:
# Per-step train loss (pre-update batch loss, all epochs). The sharp drop after
# ~1 epoch (~390 steps at 50K, batch 128) is data repeating — expected.
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
fig, ax = plt.subplots(figsize=(9, 4.5))
for (label, r), color in zip(runs.items(), colors):
    t = r["train"]
    bits = t["train_loss_nats"].map(nats_to_bits)
    ax.plot(t["step"], bits, color=color, alpha=0.25, lw=0.6)
    ax.plot(t["step"], bits.rolling(50, min_periods=1).mean(), color=color, label=label)
ax.set(xlabel="step", ylabel="train loss (bits / label token)", yscale="log",
       title="Train loss — pre-update loss on each incoming batch")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


In [ ]:
# Held-out val loss (the stopping-rule evals). "x" marks each run's last eval.
fig, ax = plt.subplots(figsize=(9, 4.5))
for (label, r), color in zip(runs.items(), colors):
    v = r["val"]
    bits = v["val_loss_nats"].map(nats_to_bits)
    stop = r["result"].get("stop_reason") or "running"
    ax.plot(v["step"], bits, color=color, marker="o", ms=3, label=f"{label} [{stop}]")
    ax.plot(v["step"].iloc[-1], bits.iloc[-1], color=color, marker="x", ms=10, mew=2)
ax.set(xlabel="step", ylabel="val loss (bits / label token)", yscale="log",
       title="Val loss — in-loop evals every eval_every steps")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


In [ ]:
# The paper-shaped curve: epoch-1 prequential loss per label token vs. examples
# seen (each point is a batch's PRE-update loss — loss on data never seen
# before). This is the panel to hold against Donoway et al.: A should start low
# (algorithm already known, only notation to remap) and flatline within few
# examples; B should trace a full learning curve.
fig, ax = plt.subplots(figsize=(9, 4.5))
for (label, r), color in zip(runs.items(), colors):
    if r["curve"] is None:
        print(f"{label}: no logs/prequential.jsonl yet (run not finished) — skipped")
        continue
    c = r["curve"]
    x = c["example_index"] + 1  # +1 so the first batch survives the log axis
    bits = c["loss_per_label_token_nats"].map(nats_to_bits)
    ax.plot(x, bits, color=color, alpha=0.25, lw=0.8)
    ax.plot(x, bits.rolling(9, min_periods=1).mean(), color=color, label=label)
ax.set(xlabel="examples seen (epoch 1)", ylabel="prequential loss (bits / label token)",
       xscale="log", yscale="log",
       title="Prequential learning curve (epoch 1) — the paper comparison")
ax.grid(alpha=0.3, which="both")
ax.legend()
plt.show()


In [ ]:
# MDL / EDL totals (bits). EDL = MDL − N·L_test needs the finalized run
# (eval/test_loss.json + masking-hash parity) — "n/a" until then.
rows = []
for label, r in runs.items():
    rid = r["id"]
    try:
        mdl_bits = nats_to_bits(mdl_nats(rid, store=STORE))
    except FileNotFoundError:
        mdl_bits = None
    try:
        edl_bits = nats_to_bits(edl_nats(rid, store=STORE))
    except Exception as e:  # incomplete run / missing test loss / hash mismatch
        edl_bits = f"n/a ({type(e).__name__})"
    rows.append({"run": label, "run_id": rid, "MDL_bits": mdl_bits, "EDL_bits": edl_bits})
pd.DataFrame(rows)


## How to read this

**Paper-consistent (what we need):**
- Arm A's prequential curve starts well below B's and flatlines within few
  examples; B traces a slower learning curve across many more examples.
- EDL_A ≪ EDL_B (elicitation reveals a latent capability cheaply; teaching pays
  for the algorithm in full).
- Both arms' val loss converges near zero — the ε/k rule fired (`stop_reason=
  converged`), so both endpoints actually learned the task.

**Trouble signs:**
1. **Curves overlap (EDL_A ≈ EDL_B)** — the run-2 algorithm pre-teach conferred
   no head start: at 38.7M there is no latent-capability contrast between the
   arms, and the elicit-vs-teach comparison has no signal to measure.
2. **A at or above B, or A not converging** — cross-notation transfer failed
   (the algorithm was taught in NL notation but doesn't carry to operator
   notation), pointing at the run-3 installer or the pre-teach itself.
3. **`stop_reason=max_steps` on either arm** — the run never converged;
   investigate before reading anything else off these plots.

Grid context: add the other `open2_b` run_ids to `RUNS` to see how B's curves
move with n — the OPEN(2) verdict itself is the G5 accuracy comparison.
